In [1]:
!pip install pyspark fastavro pandas

In [8]:
from pyspark.sql import SparkSession
import time

#  Correct avro version for Spark 4.0.2
spark = SparkSession.builder \
    .appName("Format Comparison") \
    .config("spark.jars.packages",
            "org.apache.spark:spark-avro_2.13:4.0.0") \
    .getOrCreate()

print(" Spark Ready!", spark.version)

 Spark Ready! 4.0.2


In [9]:
df = spark.read.csv(
    "/content/Tweets.csv",
    header=True,
    inferSchema=True
)
df = df.dropna(subset=["tweet_id","airline_sentiment","airline","text"])
print(" Rows loaded:", df.count())

 Rows loaded: 14632


In [10]:
import time
import fastavro
import pandas as pd
import json
import os

# Write CSV
start = time.time()
df.coalesce(1).write.csv(
    "/content/output_csv", header=True, mode="overwrite")
csv_write = round(time.time() - start, 4)
print(f" CSV write time:     {csv_write} sec")

# Write Parquet
start = time.time()
df.write.parquet("/content/output_parquet", mode="overwrite")
parquet_write = round(time.time() - start, 4)
print(f" Parquet write time: {parquet_write} sec")

#  Write Avro using fastavro (no Spark package needed)
start = time.time()
pandas_df = df.toPandas()

# Convert all columns to string to avoid type issues
pandas_df = pandas_df.astype(str)

# Build Avro schema from dataframe
fields = [{"name": col, "type": ["null", "string"], "default": None}
          for col in pandas_df.columns]
schema = {
    "type": "record",
    "name": "Tweet",
    "fields": fields
}
parsed_schema = fastavro.parse_schema(schema)

# Write Avro file
os.makedirs("/content/output_avro", exist_ok=True)
records = pandas_df.to_dict("records")
with open("/content/output_avro/tweets.avro", "wb") as f:
    fastavro.writer(f, parsed_schema, records)

avro_write = round(time.time() - start, 4)
print(f"  Avro write time:    {avro_write} sec")
print("\n All 3 formats written successfully!")

 CSV write time:     0.4878 sec
 Parquet write time: 0.5752 sec
  Avro write time:    0.8619 sec

 All 3 formats written successfully!


In [11]:
# Read CSV
start = time.time()
spark.read.csv(
    "/content/output_csv", header=True, inferSchema=True).count()
csv_read = round(time.time() - start, 4)

# Read Parquet
start = time.time()
spark.read.parquet("/content/output_parquet").count()
parquet_read = round(time.time() - start, 4)

#  Read Avro using fastavro
start = time.time()
with open("/content/output_avro/tweets.avro", "rb") as f:
    avro_records = list(fastavro.reader(f))
avro_read = round(time.time() - start, 4)

print("=" * 45)
print("     FORMAT COMPARISON RESULTS")
print("=" * 45)
print(f"{'Format':<12} {'Write (sec)':<15} {'Read (sec)'}")
print("-" * 45)
print(f"{'CSV':<12} {csv_write:<15} {csv_read}")
print(f"{'Parquet':<12} {parquet_write:<15} {parquet_read}")
print(f"{'Avro':<12} {avro_write:<15} {avro_read}")
print("=" * 45)


     FORMAT COMPARISON RESULTS
Format       Write (sec)     Read (sec)
---------------------------------------------
CSV          0.4878          0.596
Parquet      0.5752          0.3232
Avro         0.8619          0.2049


In [12]:
import os

def get_folder_size(path):
    total = 0
    for f in os.listdir(path):
        fp = os.path.join(path, f)
        if os.path.isfile(fp):
            total += os.path.getsize(fp)
    return round(total / 1024 / 1024, 2)

csv_size     = get_folder_size("/content/output_csv")
parquet_size = get_folder_size("/content/output_parquet")
avro_size    = get_folder_size("/content/output_avro")

print("=" * 40)
print("      FILE SIZE COMPARISON")
print("=" * 40)
print(f" CSV size:     {csv_size} MB")
print(f" Parquet size: {parquet_size} MB")
print(f"  Avro size:    {avro_size} MB")
print("=" * 40)
print(f" Parquet saves {round((1-parquet_size/csv_size)*100,1)}% vs CSV")
print(f" Avro saves    {round((1-avro_size/csv_size)*100,1)}% vs CSV")

      FILE SIZE COMPARISON
 CSV size:     3.27 MB
 Parquet size: 1.42 MB
  Avro size:    3.69 MB
 Parquet saves 56.6% vs CSV
 Avro saves    -12.8% vs CSV


In [13]:
import pandas as pd

summary = pd.DataFrame({
    "Format":      ["CSV", "Parquet", "Avro"],
    "Write (sec)": [csv_write, parquet_write, avro_write],
    "Read (sec)":  [csv_read, parquet_read, avro_read],
    "Size (MB)":   [csv_size, parquet_size, avro_size],
    "Best For": [
        "Compatibility, human readable",
        "Fast analytics, smallest size",
        "Streaming, schema evolution"
    ]
})

print("\n" + "="*65)
print("        FINAL FORMAT COMPARISON SUMMARY")
print("="*65)
print(summary.to_string(index=False))
print("="*65)
print("\n Fastest read:   Parquet")
print(" Smallest size:  Parquet")
print(" Best streaming: Avro")
print(" Most readable:  CSV")




        FINAL FORMAT COMPARISON SUMMARY
 Format  Write (sec)  Read (sec)  Size (MB)                      Best For
    CSV       0.4878      0.5960       3.27 Compatibility, human readable
Parquet       0.5752      0.3232       1.42 Fast analytics, smallest size
   Avro       0.8619      0.2049       3.69   Streaming, schema evolution

 Fastest read:   Parquet
 Smallest size:  Parquet
 Best streaming: Avro
 Most readable:  CSV

